# Alexandria → SLICES: conversión y auditoría

Este notebook procesa todas las estructuras de `alexandria_00000.json` con **la implementación oficial de SLICES**. La columna `slices` será la entrada textual futura de ChemBERTa; las propiedades materiales y energéticas se conservan como targets o metadata. Aquí no se tokeniza, no se entrena, no se hace un split y no se identifican polimorfos definitivos.

**Configuración reproducible**

- SLICES graph method: EconNN (`econnn`).
- SLICES strategy: `4`.
- Propósito: property prediction; la comprobación completa usa `graph_rank_check=False`.
- El CSV `all` conserva una fila por cada entry, incluidos fallos de parsing/conversión y targets ausentes. Los CSV `valid` y `energy` son vistas derivadas con los filtros explícitos de sus secciones; no se filtra por valores ausentes de otros targets ni se eliminan outliers.
- El análisis de composición es preliminar: la igualdad de composición no demuestra que dos estructuras sean polimorfos distintos.

API consultada: [repositorio oficial de SLICES](https://github.com/xiaohang007/SLICES), cuya interfaz documenta `SLICES(graph_method=...)`, `structure2SLICES(..., strategy=4)`, `check_SLICES_basic(...)` y `check_SLICES(..., graph_rank_check=False)`. El notebook también inspecciona las firmas de la instalación activa.\n\nEjecuta las celdas de arriba abajo. La conversión puede tardar; `tqdm` muestra el avance sin imprimir cada material.


## 1. Imports and configuration

Se informa el entorno activo para que queden registradas las versiones que afectan la reproducibilidad. `ijson` es opcional: si no está disponible, se usa `json.load()` de forma explícita.

In [1]:
import json
import inspect
import sys
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import pymatgen
from tqdm.auto import tqdm

from pymatgen.core import Structure
from slices.core import SLICES

print(sys.version)
print("pandas:", pd.__version__)
print("pymatgen:", getattr(pymatgen, "__version__", "versión no expuesta"))
try:
    print("SLICES:", metadata.version("slices"))
except metadata.PackageNotFoundError:
    print("SLICES: versión no disponible en importlib.metadata")
try:
    import ijson
    IJSON_AVAILABLE = True
    print("ijson: disponible (lectura incremental)")
except ImportError:
    ijson = None
    IJSON_AVAILABLE = False
    print("ijson: no disponible (se usará json.load())")


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
3.10.21 | packaged by conda-forge | (main, Aug 21 2026, 22:37:53) [Clang 19.1.7 ]
pandas: 2.3.3
pymatgen: versión no expuesta
SLICES: 2.0.12
ijson: no disponible (se usará json.load())


## 2. Input/output paths

El notebook localiza el JSON tanto si Jupyter se inició desde el proyecto como desde `data/notebooks_processing`. Los CSV se escriben junto a este notebook en `data/notebooks_processing/`; el JSON original se abre solo para lectura.

In [2]:
INPUT_FILENAME = "alexandria_00000.json"
SEARCH_BASES = [Path.cwd(), *Path.cwd().parents]
input_candidates = []
for base in SEARCH_BASES:
    input_candidates.extend([
        base / "data" / "raw_data" / INPUT_FILENAME,
        base / "raw_data" / INPUT_FILENAME,
    ])
INPUT_PATH = next((candidate.resolve() for candidate in input_candidates if candidate.is_file()), None)
if INPUT_PATH is None:
    raise FileNotFoundError(
        f"No se encontró {INPUT_FILENAME}. Directorio actual: {Path.cwd()}"
    )
OUTPUT_DIR = INPUT_PATH.parent.parent / "notebooks_processing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_CSV = OUTPUT_DIR / "alexandria_00000_slices_all.csv"
VALID_CSV = OUTPUT_DIR / "alexandria_00000_slices_valid.csv"
ENERGY_CSV = OUTPUT_DIR / "alexandria_00000_slices_energy.csv"

print("Input:", INPUT_PATH)
print("Tamaño del JSON (MB):", round(INPUT_PATH.stat().st_size / 1_000_000, 2))
print("Output directory:", OUTPUT_DIR)


Input: /Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/raw_data/alexandria_00000.json
Tamaño del JSON (MB): 403.42
Output directory: /Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/notebooks_processing


## 3. Load Alexandria dataset

Con `ijson` se hace una pasada para contar `entries.item` sin guardar todas las estructuras en memoria y luego se vuelve a recorrer el archivo al procesarlo. Sin `ijson`, el fallback carga el JSON una sola vez y reutiliza su lista `entries`.

In [3]:
loaded_data = None
if IJSON_AVAILABLE:
    with INPUT_PATH.open("rb") as json_file:
        total_entries = sum(1 for _ in ijson.items(json_file, "entries.item"))
else:
    with INPUT_PATH.open("r", encoding="utf-8") as json_file:
        loaded_data = json.load(json_file)
    if not isinstance(loaded_data, dict) or not isinstance(loaded_data.get("entries"), list):
        raise ValueError("El JSON debe contener una lista en data['entries'].")
    total_entries = len(loaded_data["entries"])

print(f"Total Alexandria entries: {total_entries}")

def iter_entries():
    """Itera entries sin retener el dataset completo si ijson está disponible."""
    if IJSON_AVAILABLE:
        with INPUT_PATH.open("rb") as json_file:
            yield from ijson.items(json_file, "entries.item")
    else:
        yield from loaded_data["entries"]


Total Alexandria entries: 100000


## 4. Initialize SLICES

La instancia usa EconNN y desactiva la carga de M3GNet (`relax_model=None`): este notebook solo codifica y valida estructuras, y no relaja ni reconstruye cristales. Así se evita cargar un modelo TensorFlow/M3GNet que no participa en la conversión y que puede fallar con Keras 3. Se muestra la firma instalada de los métodos oficiales antes de procesar el dataset.

In [5]:
GRAPH_METHOD = "econnn"
STRATEGY = 4
backend_kwargs = {"graph_method": GRAPH_METHOD}
init_signature = inspect.signature(SLICES).parameters
if "relax_model" in init_signature:
    backend_kwargs["relax_model"] = None
backend = SLICES(**backend_kwargs)

print("SLICES graph method:", backend.graph_method)
print("M3GNet relaxer loaded:", hasattr(backend, "relaxer"))
print("structure2SLICES signature:", inspect.signature(backend.structure2SLICES))
print("check_SLICES_basic signature:", inspect.signature(backend.check_SLICES_basic))
if hasattr(backend, "check_SLICES"):
    print("check_SLICES signature:", inspect.signature(backend.check_SLICES))


SLICES graph method: econnn
M3GNet relaxer loaded: False
structure2SLICES signature: (structure, strategy=4)
check_SLICES_basic signature: (SLICES, strategy=4)
check_SLICES signature: (SLICES, strategy=4, dupli_check=False, graph_rank_check=True)


## 5. Helper functions

Cada entry produce un diccionario con el mismo esquema. Los errores de reconstrucción, codificación y validación se capturan por separado para poder diagnosticar cada etapa. Los metadatos complejos se convierten a JSON/string antes de llegar al CSV.

In [6]:
OUTPUT_COLUMNS = [
    "mat_id", "prototype_id", "formula", "reduced_formula", "composition_key",
    "composition", "elements", "n_elements", "spg", "nsites", "volume",
    "volume_per_atom", "slices", "slices_basic_valid", "slices_valid",
    "energy", "energy_total", "energy_corrected", "energy_per_atom",
    "energy_corrected_per_atom", "e_form", "e_above_hull", "e_phase_separation",
    "correction", "band_gap_ind", "band_gap_dir", "total_mag", "dos_ef",
    "stress", "decomposition", "location", "run_timestamp",
    "conversion_success", "conversion_error", "structure_parsing_error",
    "slices_conversion_error", "slices_validation_error",
]

NUMERIC_FIELDS = (
    "energy", "energy_total", "energy_corrected", "e_form", "e_above_hull",
    "e_phase_separation", "correction", "band_gap_ind", "band_gap_dir",
    "total_mag", "dos_ef", "spg", "nsites",
)

def call_with_strategy(method, *args, **extra_kwargs):
    """Invoca una función oficial pasando strategy=4 según su firma instalada."""
    parameters = inspect.signature(method).parameters
    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())
    if "strategy" in parameters or accepts_kwargs:
        return method(*args, strategy=STRATEGY, **extra_kwargs)
    positional = [
        p for p in parameters.values()
        if p.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD)
    ]
    if len(positional) > len(args):
        return method(*args, STRATEGY, **extra_kwargs)
    raise TypeError(f"{method.__name__} no expone un argumento strategy compatible: {inspect.signature(method)}")

def csv_string(value):
    """Devuelve texto apto para CSV; mantiene strings simples sin comillas JSON."""
    if value is None:
        return None
    if isinstance(value, str):
        return value
    try:
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
    except (TypeError, ValueError):
        return str(value)

def numeric_or_none(value):
    """Normaliza escalares numéricos; devuelve None para valores ausentes/no numéricos."""
    if value is None or isinstance(value, (dict, list, tuple)):
        return None
    try:
        result = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return result if np.isfinite(result) else None

def first_present(primary, secondary, key):
    value = primary.get(key) if isinstance(primary, dict) else None
    return value if value is not None else (secondary.get(key) if isinstance(secondary, dict) else None)

def empty_record(entry):
    """Extrae campos disponibles incluso si la estructura está dañada."""
    data = entry.get("data") or {}
    if not isinstance(data, dict):
        data = {}
    record = {column: None for column in OUTPUT_COLUMNS}
    for key in NUMERIC_FIELDS:
        record[key] = numeric_or_none(first_present(data, entry, key))
    for key in ("mat_id", "prototype_id", "formula", "decomposition", "location", "run_timestamp"):
        value = first_present(data, entry, key)
        record[key] = csv_string(value)
    record["correction"] = numeric_or_none(entry.get("correction", data.get("correction")))
    record["composition"] = csv_string(entry.get("composition"))
    record["elements"] = None
    record["stress"] = csv_string(data.get("stress"))
    record["conversion_success"] = False
    return record

def process_entry(entry, backend):
    """Reconstruye una estructura y genera/valida SLICES con la API oficial."""
    record = empty_record(entry if isinstance(entry, dict) else {})
    if not isinstance(entry, dict):
        message = f"Se esperaba un objeto entry; recibido {type(entry).__name__}."
        record["structure_parsing_error"] = message
        record["conversion_error"] = message
        return record

    structure_dict = entry.get("structure")
    try:
        if not isinstance(structure_dict, dict):
            raise ValueError("entry['structure'] no contiene un diccionario serializado.")
        structure = Structure.from_dict(structure_dict)
        record["reduced_formula"] = structure.composition.reduced_formula
        record["composition_key"] = structure.composition.reduced_composition.alphabetical_formula
        record["elements"] = csv_string(sorted({element.symbol for element in structure.composition.elements}))
        record["n_elements"] = len(structure.composition.elements)
        record["nsites"] = len(structure)
        record["volume"] = numeric_or_none(structure.volume)
        record["volume_per_atom"] = numeric_or_none(structure.volume / len(structure)) if len(structure) else None
        if record["composition"] is None:
            record["composition"] = csv_string(structure.composition.as_dict())
    except Exception as exc:
        record["structure_parsing_error"] = f"{type(exc).__name__}: {exc}"
        record["conversion_error"] = record["structure_parsing_error"]
        return record

    # Si falta el dato energy, se preserva el energy de nivel superior de ComputedStructureEntry.
    record["energy"] = numeric_or_none(entry.get("energy", record["energy"]))
    atom_count = len(structure)
    record["energy_per_atom"] = (
        record["energy"] / atom_count if record["energy"] is not None and atom_count else None
    )
    record["energy_corrected_per_atom"] = (
        record["energy_corrected"] / atom_count
        if record["energy_corrected"] is not None and atom_count else None
    )

    try:
        slices_string = call_with_strategy(backend.structure2SLICES, structure)
        if not isinstance(slices_string, str) or not slices_string.strip():
            raise ValueError("SLICES devolvió una cadena vacía o un valor no textual.")
        record["slices"] = slices_string.strip()
        record["conversion_success"] = True
    except Exception as exc:
        record["slices_conversion_error"] = f"{type(exc).__name__}: {exc}"
        record["conversion_error"] = record["slices_conversion_error"]
        return record

    try:
        record["slices_basic_valid"] = bool(
            call_with_strategy(backend.check_SLICES_basic, record["slices"])
        )
    except Exception as exc:
        record["slices_validation_error"] = f"check_SLICES_basic: {type(exc).__name__}: {exc}"

    if hasattr(backend, "check_SLICES"):
        try:
            check_signature = inspect.signature(backend.check_SLICES)
            parameters = check_signature.parameters
            check_kwargs = {}
            if "graph_rank_check" in parameters:
                check_kwargs["graph_rank_check"] = False
            elif any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values()):
                check_kwargs["graph_rank_check"] = False
            record["slices_valid"] = bool(
                call_with_strategy(backend.check_SLICES, record["slices"], **check_kwargs)
            )
        except Exception as exc:
            record["slices_validation_error"] = (
                f"check_SLICES: {type(exc).__name__}: {exc}"
            )
    else:
        record["slices_validation_error"] = "La versión instalada no expone check_SLICES."

    return record


## 6. Test conversion on one structure

Antes del recorrido completo se reconstruye y convierte una entry usando exactamente `process_entry`. Luego se muestran el objeto `Structure`, la cadena y los resultados de validación. Si falla la conversión de esta primera entry, el procesamiento global seguirá registrando errores; revisa aquí el entorno y el mensaje antes de lanzar un procesamiento largo.

In [7]:
first_entry = next(iter(iter_entries()), None)
if first_entry is None:
    raise ValueError("El archivo no contiene entries para convertir.")

test_record = process_entry(first_entry, backend)
if isinstance(first_entry.get("structure"), dict):
    try:
        test_structure = Structure.from_dict(first_entry["structure"])
    except Exception:
        test_structure = None
else:
    test_structure = None

print("structure:")
print(test_structure)
print("slices_string:")
print(test_record["slices"])
print("validation_result:", {
    "slices_basic_valid": test_record["slices_basic_valid"],
    "slices_valid": test_record["slices_valid"],
    "conversion_error": test_record["conversion_error"],
    "slices_validation_error": test_record["slices_validation_error"],
})


structure:
Full Formula (Sr1 Cd10 Rh2)
Reduced Formula: Sr(Cd5Rh)2
abc   :   5.816792   7.682453   7.534837
angles:  81.597395  67.294580  67.754460
pbc   :       True       True       True
Sites (13)
  #  SP           a         b         c    charge  forces                                    magmom
---  ----  --------  --------  --------  --------  --------------------------------------  --------
  0  Sr    0         0         0            7.974  [0.0, -0.0, -0.0]                             -0
  1  Cd    0.34055   0.639921  0.678977    10.308  [-0.00104236, -0.00284227, 0.00255666]         0
  2  Cd    0.65945   0.360079  0.321023    10.308  [0.00104236, 0.00284227, -0.00255666]          0
  3  Cd    0.980472  0.360079  0.678977    10.308  [-0.00104236, 0.00284227, 0.00255666]          0
  4  Cd    0.019528  0.639921  0.321023    10.308  [0.00104236, -0.00284227, -0.00255666]         0
  5  Cd    0         0         0.5         10.335  [0.0, -0.0, -0.0]                             -0

## 7. Convert all Alexandria entries

Se recorre el archivo con progreso. La primera entry reutiliza su resultado de prueba para evitar repetir una conversión costosa. Una fila se añade por entry, también si la estructura no se puede reconstruir o si SLICES falla.

In [8]:
records = []
for index, entry in enumerate(tqdm(iter_entries(), total=total_entries, desc="Alexandria → SLICES")):
    record = test_record if index == 0 else process_entry(entry, backend)
    records.append(record)

df = pd.DataFrame.from_records(records, columns=OUTPUT_COLUMNS)
assert len(df) == total_entries, f"Esperadas {total_entries} filas; obtenidas {len(df)}."
print("Filas del DataFrame:", len(df))


Alexandria → SLICES:   0%|          | 0/100000 [00:00<?, ?it/s]

ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!


ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!


...

ERROR - could not find a linearly independent cocycle basis!


ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!


ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not

Filas del DataFrame: 100000


## 8. Build DataFrame

`df` es la tabla de auditoría completa y mantiene una fila por cada entry del JSON. Los errores de parsing/conversión/validación están separados; `conversion_error` resume los errores que impidieron reconstruir la estructura o crear la cadena. Ninguna fila se elimina por un target secundario ausente.

In [9]:
display(df.head(3))
print("Columnas faltantes:", sorted(set(OUTPUT_COLUMNS) - set(df.columns)))
print("Filas conservadas:", len(df), "de", total_entries)


,mat_id,prototype_id,formula,reduced_formula,composition_key,composition,elements,n_elements,spg,nsites,...,dos_ef,stress,decomposition,location,run_timestamp,conversion_success,conversion_error,structure_parsing_error,slices_conversion_error,slices_validation_error
0,agm005737469,AB2C10_0_spg69,Sr(Cd5Rh)2,Sr(Cd5Rh)2,Cd10 Rh2 Sr1,"{""Cd"": 10.0, ""Rh"": 2.0, ""Sr"": 1.0}","[""Cd"", ""Rh"", ""Sr""]",3,69.0,13,...,6.497900,"[[7.6139755, 5.1e-07, 0.14952564], [5.1e-07, 7...",Cd41Rh8 Sr3Cd11 SrCd3Rh2,cgat_comp_2/ternaries/AB2C10/runs/batch-001/Sr...,"{""@class"": ""datetime"", ""@module"": ""datetime"", ...",True,None,None,None,None
1,agm002510391,ABC3_0_spg221,Li3CaPt,Li3CaPt,Ca1 Li3 Pt1,"{""Ca"": 1.0, ""Li"": 3.0, ""Pt"": 1.0}","[""Ca"", ""Li"", ""Pt""]",3,221.0,5,...,2.520410,"[[0.9447303, 0.0, 0.0], [0.0, 0.9447303, 0.0],...",Li2CaPt Li,high_throughput/perovskites/Li/Li3CaPt/xxx_02p...,"{""@class"": ""datetime"", ""@module"": ""datetime"", ...",True,None,None,None,None
2,agm004513860,AB2C3D4_7_spg115,Nd2TiGa4Pt3,Nd2TiGa4Pt3,Ga4 Nd2 Pt3 Ti1,"{""Ga"": 4.0, ""Nd"": 2.0, ""Pt"": 3.0, ""Ti"": 1.0}","[""Ga"", ""Nd"", ""Pt"", ""Ti""]",4,115.0,10,...,5.601462,"[[3.1229622, 0.0, 0.0], [0.0, 3.1229622, 0.0],...",Nd2Ga5Pt3 NdGaPt TiGaPt NdGa2,cgat_comp/quaternaries/AB2C3D4/runs/batch-000/...,"{""@class"": ""datetime"", ""@module"": ""datetime"", ...",True,None,None,None,None


Columnas faltantes: []
Filas conservadas: 100000 de 100000


## 9. Validate SLICES

Se inspeccionan los resultados de ambas comprobaciones. `conversion_success` indica que la librería produjo una cadena no vacía; las columnas de validación indican por separado si esa cadena supera cada comprobación. Una validación no soportada por una versión instalada queda como valor ausente y con detalle en `slices_validation_error`.

In [10]:
print("Conversión exitosa:", int(df["conversion_success"].fillna(False).sum()))
print("SLICES basic válidas:", int(df["slices_basic_valid"].eq(True).sum()))
print("SLICES completas válidas:", int(df["slices_valid"].eq(True).sum()))
print("Cadenas SLICES ausentes:", int(df["slices"].isna().sum()))


Conversión exitosa: 99998
SLICES basic válidas: 99998
SLICES completas válidas: 98394
Cadenas SLICES ausentes: 2


## 10. Energy targets and metadata

Se conservan las energías de entrada (`energy`, `energy_total`, `energy_corrected`) junto a `energy_per_atom` y `energy_corrected_per_atom`. Los valores corregidos por átomo se derivan solo de `energy_corrected`; nunca se sustituye una energía corregida ausente por la energía sin corregir. `stress`, `decomposition`, `location` y `run_timestamp` se serializan como texto apto para CSV.

In [11]:
energy_target_columns = [
    "energy", "energy_total", "energy_corrected", "energy_per_atom",
    "energy_corrected_per_atom", "e_form", "e_above_hull", "e_phase_separation",
    "band_gap_ind", "band_gap_dir", "total_mag", "dos_ef",
]
print("Targets conservados:", energy_target_columns)
print("NaN por target:\n", df[energy_target_columns].isna().sum())


Targets conservados: ['energy', 'energy_total', 'energy_corrected', 'energy_per_atom', 'energy_corrected_per_atom', 'e_form', 'e_above_hull', 'e_phase_separation', 'band_gap_ind', 'band_gap_dir', 'total_mag', 'dos_ef']
NaN por target:
 energy                       0
energy_total                 0
energy_corrected             0
energy_per_atom              0
energy_corrected_per_atom    0
e_form                       0
e_above_hull                 0
e_phase_separation           0
band_gap_ind                 0
band_gap_dir                 0
total_mag                    0
dos_ef                       0
dtype: int64


## 11. Save full CSV

Este CSV es el artefacto de auditoría principal: incluye todos los registros, incluso filas sin SLICES y filas con targets nulos. Las columnas `composition`, `elements`, `stress` y `decomposition` son strings; el resto de metadata no usa objetos Python complejos.

In [12]:
df.to_csv(ALL_CSV, index=False, encoding="utf-8")
print("Guardado:", ALL_CSV)


Guardado: /Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/notebooks_processing/alexandria_00000_slices_all.csv


## 12. Create valid-SLICES dataset

Se aplica únicamente el criterio explícito para la vista inicial de ML: conversión exitosa, validación básica verdadera y cadena no nula. No se filtra por target ausente, `NaN` en otras propiedades ni por valores extremos. Los resultados de validación completa se conservan como columnas informativas.

In [13]:
df_valid = df[
    (df["conversion_success"] == True)
    & (df["slices_basic_valid"] == True)
    & (df["slices"].notna())
].copy()
df_valid.to_csv(VALID_CSV, index=False, encoding="utf-8")
print(f"Filas con SLICES según el criterio declarado: {len(df_valid)} / {len(df)}")
print("Guardado:", VALID_CSV)


Filas con SLICES según el criterio declarado: 99998 / 100000
Guardado: /Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/notebooks_processing/alexandria_00000_slices_valid.csv


## 13. Create energy dataset

`df_energy` parte de `df_valid` y requiere un valor finito de `energy_corrected_per_atom`. Una `energy` disponible no se usa para rellenar `energy_corrected`. Los demás targets ausentes se mantienen sin cambios.

In [14]:
corrected_per_atom_numeric = pd.to_numeric(
    df_valid["energy_corrected_per_atom"], errors="coerce"
)
df_energy = df_valid.loc[
    corrected_per_atom_numeric.notna()
    & np.isfinite(corrected_per_atom_numeric)
].copy()
df_energy.to_csv(ENERGY_CSV, index=False, encoding="utf-8")
print(f"Filas con SLICES y energía corregida/átomo finita: {len(df_energy)} / {len(df_valid)}")
print("Guardado:", ENERGY_CSV)


Filas con SLICES y energía corregida/átomo finita: 99998 / 99998
Guardado: /Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/notebooks_processing/alexandria_00000_slices_energy.csv


## 14. Dataset quality checks

Los chequeos reportan duplicados y distribuciones sin modificar el dataset. Las estadísticas numéricas usan coerción solo para el reporte; no se eliminan observaciones ni se winsorizan outliers.

In [15]:
print("mat_id duplicados (incluye IDs nulos repetidos=False):", int(df["mat_id"].duplicated().sum()))
print("SLICES duplicados no nulos:", int(df["slices"].dropna().duplicated().sum()))
print("NaN por target:\n", df[energy_target_columns].isna().sum())

for column in ("nsites", "spg", "n_elements"):
    print(f"\nDistribución de {column}:")
    display(df[column].value_counts(dropna=False).sort_index())

for column in (
    "energy_corrected_per_atom", "energy_per_atom", "e_form", "e_above_hull",
    "e_phase_separation", "band_gap_ind", "band_gap_dir", "total_mag", "dos_ef",
):
    values = pd.to_numeric(df[column], errors="coerce")
    finite_values = values[np.isfinite(values)]
    print(f"\n{column} (valores finitos):")
    display(finite_values.describe())


mat_id duplicados (incluye IDs nulos repetidos=False): 0
SLICES duplicados no nulos: 0
NaN por target:
 energy                       0
energy_total                 0
energy_corrected             0
energy_per_atom              0
energy_corrected_per_atom    0
e_form                       0
e_above_hull                 0
e_phase_separation           0
band_gap_ind                 0
band_gap_dir                 0
total_mag                    0
dos_ef                       0
dtype: int64

Distribución de nsites:


nsites
1          7
2        143
3       2802
4      21021
5      13548
       ...  
140        5
144        3
148        1
150        2
152        1
Name: count, Length: 103, dtype: int64


Distribución de spg:


spg
1.0      1501
2.0      1598
3.0        55
4.0       130
5.0       356
         ... 
225.0    3467
226.0       9
227.0     267
229.0     272
230.0     268
Name: count, Length: 196, dtype: int64


Distribución de n_elements:


...

count    100000.000000
mean          0.207952
std           0.725503
min           0.000000
25%           0.000000
50%           0.000100
75%           0.019800
max           8.957500
Name: band_gap_dir, dtype: float64


total_mag (valores finitos):


count    1.000000e+05
mean     1.369290e+00
std      4.374018e+00
min     -5.500000e+01
25%     -1.000000e-07
50%      1.400000e-06
75%      9.658772e-01
max      1.443637e+02
Name: total_mag, dtype: float64


dos_ef (valores finitos):


count    100000.000000
mean          6.827096
std           8.218216
min          -4.320755
25%           2.444933
50%           4.533049
75%           8.388111
max         268.545070
Name: dos_ef, dtype: float64

## 15. Composition/polymorph preliminary analysis

El conteo agrupa las filas de `df_energy` por `composition_key`. Esto identifica composiciones repetidas para análisis posterior, pero **no** confirma polimorfos: antes de filtrar será necesario comparar estructuras y eliminar duplicados con `StructureMatcher`. Aquí no se aplica el corte de dos o más estructuras.

In [16]:
counts = (
    df_energy
    .groupby("composition_key")
    .size()
    .sort_values(ascending=False)
)
unique_compositions = int(counts.size)
print("Número de composiciones únicas:", unique_compositions)
print("Número de composiciones con 1 estructura:", int((counts == 1).sum()))
print("Número de composiciones con >= 2 estructuras:", int((counts >= 2).sum()))
print("Número de composiciones con >= 3 estructuras:", int((counts >= 3).sum()))
print("Número de composiciones con >= 5 estructuras:", int((counts >= 5).sum()))
print("Máximo número de estructuras para una composición:", int(counts.max()) if len(counts) else 0)
display(counts.head(20).rename("n_structures").to_frame())


Número de composiciones únicas: 96588
Número de composiciones con 1 estructura: 93507
Número de composiciones con >= 2 estructuras: 3081
Número de composiciones con >= 3 estructuras: 255
Número de composiciones con >= 5 estructuras: 12
Máximo número de estructuras para una composición: 12


,n_structures
composition_key,
B1 C1,12
B7 C5,11
B5 C7,8
Co5 Li9 Mn2 O16,7
O2 Si1,7
Hg1,7
Cd1 Cl2 La1,5
Sm1 Tb1 Tm2,5
B2 C1,5


## 16. Final summary

Se imprimen tasas respecto al total del JSON. La tasa de retención de energía usa como numerador las filas del dataset energético (`df_energy`), que además satisfacen el criterio de SLICES válido declarado en el notebook. Las excepciones más frecuentes ayudan a priorizar la inspección manual.

In [17]:
n_total = len(df)
n_structure_parsed = int(df["structure_parsing_error"].isna().sum())
n_conversion_success = int(df["conversion_success"].fillna(False).sum())
n_conversion_failed = n_total - n_conversion_success
n_basic_valid = int(df["slices_basic_valid"].eq(True).sum())
n_invalid_or_not_basic = n_total - n_basic_valid
n_energy_valid = len(df_energy)

def pct(numerator, denominator):
    return 100.0 * numerator / denominator if denominator else 0.0

print("========================================")
print("ALEXANDRIA → SLICES CONVERSION SUMMARY")
print("========================================")
print(f"Total entries:                  {n_total:>8}")
print(f"Structure.from_dict successful: {n_structure_parsed:>8}")
print(f"SLICES conversion successful:   {n_conversion_success:>8}")
print(f"SLICES conversion failed:       {n_conversion_failed:>8}")
print(f"SLICES basic valid:             {n_basic_valid:>8}")
print(f"SLICES invalid:                 {n_invalid_or_not_basic:>8}")
print(f"Valid SLICES with energy:       {n_energy_valid:>8}")
print()
print(f"Conversion success rate: {pct(n_conversion_success, n_total):.2f} %")
print(f"Valid SLICES rate:        {pct(n_basic_valid, n_total):.2f} %")
print(f"Energy retention rate:    {pct(n_energy_valid, n_total):.2f} %")
print()
print("Errores de conversión más frecuentes:")
display(
    df.loc[~df["conversion_success"].fillna(False), "conversion_error"]
      .value_counts(dropna=True)
      .head(10)
)
print("\nErrores de validación más frecuentes:")
display(df["slices_validation_error"].value_counts(dropna=True).head(10))


ALEXANDRIA → SLICES CONVERSION SUMMARY
Total entries:                    100000
Structure.from_dict successful:   100000
SLICES conversion successful:      99998
SLICES conversion failed:              2
SLICES basic valid:                99998
SLICES invalid:                        2
Valid SLICES with energy:          99998

Conversion success rate: 100.00 %
Valid SLICES rate:        100.00 %
Energy retention rate:    100.00 %

Errores de conversión más frecuentes:


conversion_error
ValueError: min() arg is an empty sequence    2
Name: count, dtype: int64


Errores de validación más frecuentes:


Series([], Name: count, dtype: int64)

### Revisión de ejemplos

Se muestran hasta cinco conversiones exitosas con los campos solicitados y, cuando existan, hasta cinco filas con error de conversión. Estos ejemplos son solo para inspección: no afectan los CSV.

In [18]:
successful_examples = df.loc[
    df["conversion_success"].fillna(False),
    ["mat_id", "formula", "spg", "nsites", "slices",
     "energy_corrected_per_atom", "e_form", "e_above_hull"],
].head(5)
print("Primeras 5 conversiones exitosas:")
display(successful_examples)

failed_examples = df.loc[
    ~df["conversion_success"].fillna(False),
    ["mat_id", "formula", "structure_parsing_error", "slices_conversion_error", "conversion_error"],
].head(5)
print("Hasta 5 conversiones fallidas:")
display(failed_examples)

print("\nArchivos producidos:")
for output_path in (ALL_CSV, VALID_CSV, ENERGY_CSV):
    print(f"- {output_path} ({output_path.stat().st_size / 1_000_000:.2f} MB)")


Primeras 5 conversiones exitosas:


,mat_id,formula,spg,nsites,slices,energy_corrected_per_atom,e_form,e_above_hull
0,agm005737469,Sr(Cd5Rh)2,69.0,13,+ v a ODD a DOD b OOO c OOO o Sr Cd Cd Cd Cd C...,-2.117578,-0.146191,0.048584
1,agm002510391,Li3CaPt,221.0,5,+ v b OOO c OOO d OOO e OOO o Ca Pt Li Li Li 0...,-2.980232,-0.213921,0.223922
2,agm004513860,Nd2TiGa4Pt3,115.0,10,o w b OOO m OOO j OOO o Nd Nd Ti Ga Ga Ga Ga P...,-5.630619,-0.848516,0.057553
3,agm003886200,SrMnBr2,12.0,4,+ x a DDO c OOO o Br Br Mn Sr 0 2 oo- 0 2 +o- ...,-4.206092,-0.654886,0.938851
4,agm034649907,Na2Tb3SmSe6,12.0,12,+ x a DDO c OOO o Na Na Tb Tb Tb Sm Se Se Se S...,-5.248430,-1.731583,0.006221


Hasta 5 conversiones fallidas:


,mat_id,formula,structure_parsing_error,slices_conversion_error,conversion_error
6643,agm080032697,Pd,None,ValueError: min() arg is an empty sequence,ValueError: min() arg is an empty sequence
82427,agm080032905,Ce,None,ValueError: min() arg is an empty sequence,ValueError: min() arg is an empty sequence



Archivos producidos:


FileNotFoundError: [Errno 2] No such file or directory: '/Users/cristellemadrid/Desktop/Transformers/transformers-transfer-fine-tuning-thesis/data/notebooks_processing/alexandria_00000_slices_all.csv'